# 🏀 NBA Player & Team Statistics (1980–2026) — Complete EDA

**50K player-seasons · 1,410 team-seasons · 46 years · Legends + Advanced Metrics**

> *"Basketball is a beautiful game when the five players on the court play with one heartbeat."* — Dean Smith

1. Overview | 2. Era Analysis | 3. Scoring Evolution | 4. Team Win Patterns
5. Legend Careers | 6. Advanced Metrics | 7. Salary Inflation | 8. 3-Point Revolution
9. Position Analysis | 10. All-Star Intelligence | 11. Conference Trends | 12. Win Predictor

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import matplotlib.patches as mpatches
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi']=110
plt.rcParams['axes.facecolor']='#17181c'; plt.rcParams['figure.facecolor']='#17181c'
plt.rcParams['text.color']='white'; plt.rcParams['axes.labelcolor']='white'
plt.rcParams['xtick.color']='white'; plt.rcParams['ytick.color']='white'
plt.rcParams['axes.edgecolor']='#2a2d38'; plt.rcParams['grid.color']='#202228'

ORANGE='#F77F00'; BLUE='#0077B6'; RED='#E63946'; GOLD='#FFD700'
GREEN='#52B788'; PURPLE='#7B2D8B'; TEAL='#2EC4B6'; SMOKE='#8D99AE'
ERA_COLORS={'Showtime/Bird Era':GOLD,'Jordan Era':RED,'Post-Jordan/Shaq':PURPLE,
            'LeBron Era':BLUE,'Warriors Dynasty':GREEN,'Bubble/Post-Covid':TEAL,'Modern Era':ORANGE}
POS_COLORS={'PG':BLUE,'SG':ORANGE,'SF':RED,'PF':GREEN,'C':PURPLE}
print("✅ Ready — Tip-off!")

## 1. Load & Overview

In [ ]:
INPUT="/kaggle/input/nba-player-team-statistics-1980-2026"
players=pd.read_csv(f"{INPUT}/player_season_stats.csv")
teams=pd.read_csv(f"{INPUT}/team_season_records.csv")
careers=pd.read_csv(f"{INPUT}/career_profiles.csv")
eras=pd.read_csv(f"{INPUT}/era_summary.csv")

print(f"Player-seasons: {len(players):,} | Team-seasons: {len(teams):,}")
print(f"Players:  {players['player_id'].nunique():,} | Seasons: {players['season'].min()}–{players['season'].max()}")
print(f"All-Stars: {players['all_star'].sum():,} | Avg PPG: {players['points_per_game'].mean():.1f}")
players[players['is_legend']==1].head(3)

## 2. How Basketball Changed — Era Analysis

In [ ]:
era_order=['Showtime/Bird Era','Jordan Era','Post-Jordan/Shaq','LeBron Era','Warriors Dynasty','Bubble/Post-Covid','Modern Era']
era_plot=eras.set_index('era').reindex(era_order).dropna()
metrics=[('avg_pace','Avg Pace'),('avg_pts_for','Avg Team Points'),('avg_three_pct','3P Attempt %'),
         ('avg_ts_pct','True Shooting %'),('avg_payroll_m','Avg Payroll ($M)'),('avg_attendance','Home Attendance')]

fig,axes=plt.subplots(2,3,figsize=(18,10))
for ax,(col,label) in zip(axes.flatten(),metrics):
    colors_e=[ERA_COLORS.get(e,SMOKE) for e in era_plot.index]
    era_plot[col].plot.bar(ax=ax,color=colors_e,edgecolor='none',alpha=0.9)
    ax.set_title(label,fontweight='bold',color='white',fontsize=11)
    ax.tick_params(axis='x',rotation=35); ax.set_xlabel('')
plt.suptitle('How the NBA Changed Across 7 Eras',fontsize=14,fontweight='bold',color='white',y=1.01)
plt.tight_layout(); plt.show()

## 3. Scoring & Efficiency Evolution

In [ ]:
season_avg=players.groupby('season').agg(avg_pts=('points_per_game','mean'),avg_per=('player_efficiency_rating','mean'),
    avg_fg=('fg_pct','mean'),avg_fg3=('fg3_pct','mean')).reset_index()

fig,axes=plt.subplots(1,2,figsize=(16,6))
axes[0].plot(season_avg['season'],season_avg['avg_pts'],color=ORANGE,linewidth=2.2,marker='o',markersize=3,label='PPG')
ax2=axes[0].twinx()
ax2.plot(season_avg['season'],season_avg['avg_per'],color=BLUE,linewidth=2,linestyle='--',marker='s',markersize=3,label='PER')
axes[0].set_title('League Avg PPG & PER Over Time',fontweight='bold',color='white')
axes[0].set_ylabel('PPG',color=ORANGE); ax2.set_ylabel('PER',color=BLUE)
axes[0].legend(loc='upper left',fontsize=9); ax2.legend(loc='lower right',fontsize=9)

axes[1].plot(season_avg['season'],season_avg['avg_fg']*100,color=GREEN,linewidth=2,label='FG%')
axes[1].plot(season_avg['season'],season_avg['avg_fg3']*100,color=ORANGE,linewidth=2,label='3P%')
axes[1].set_title('FG% & 3-Point% Trends',fontweight='bold',color='white')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## 4. Team Win Patterns

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(18,8))
team_wins=teams.groupby('team_name')['wins'].mean().sort_values(ascending=True).tail(20)
[BLUE,ORANGE,RED]
colors_w=[RED if w>50 else ORANGE if w>45 else BLUE for w in team_wins.values]
team_wins.plot.barh(ax=axes[0],color=colors_w,edgecolor='none',alpha=0.9)
axes[0].axvline(41,color='white',linestyle='--',alpha=0.5,label='Playoff threshold')
axes[0].set_title('Top 20 Teams by Avg Wins per Season',fontsize=13,fontweight='bold',color='white')
axes[0].legend(fontsize=9)

axes[1].scatter(teams['net_rating'],teams['win_pct'],
    c=[list(ERA_COLORS.keys()).index(e) if e in ERA_COLORS else 0 for e in teams['era']],
    cmap='tab10',alpha=0.4,s=15,edgecolors='none')
corr_nr=teams['net_rating'].corr(teams['win_pct'])
axes[1].set_title(f'Net Rating vs Win % (r={corr_nr:.3f})',fontweight='bold',color='white')
axes[1].set_xlabel('Net Rating'); axes[1].set_ylabel('Win %')
axes[1].grid(True,alpha=0.15)
plt.tight_layout(); plt.show()

## 5. Legend Career Arcs

In [ ]:
legends_df=players[players['is_legend']==1]
top10=["Michael Jordan","LeBron James","Kobe Bryant","Stephen Curry","Shaquille O'Neal",
       "Tim Duncan","Magic Johnson","Giannis Antetokounmpo","Kevin Durant","Nikola Jokic"]
colors10=[RED,ORANGE,GOLD,GREEN,PURPLE,TEAL,BLUE,ORANGE,RED,GREEN]

fig,axes=plt.subplots(1,2,figsize=(16,7))
for name,col in zip(top10,colors10):
    d=legends_df[legends_df['player_name']==name].sort_values('age')
    if len(d)>2:
        axes[0].plot(d['age'],d['points_per_game'],label=name.split()[-1],color=col,linewidth=1.8,alpha=0.85,marker='o',markersize=3)
axes[0].set_title('Scoring Career Arcs (PPG by Age)',fontsize=13,fontweight='bold',color='white')
axes[0].set_xlabel('Age'); axes[0].legend(fontsize=7,ncol=2); axes[0].grid(True,alpha=0.15)

for name,col in zip(top10[:6],colors10):
    d=legends_df[legends_df['player_name']==name].sort_values('age')
    if len(d)>2:
        axes[1].plot(d['age'],d['player_efficiency_rating'],label=name.split()[-1],color=col,linewidth=1.8,alpha=0.85)
axes[1].set_title('PER Career Arcs',fontsize=13,fontweight='bold',color='white')
axes[1].set_xlabel('Age'); axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.15)
plt.tight_layout(); plt.show()

## 6. Advanced Metrics Deep-Dive

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(16,10))

players['player_efficiency_rating'].clip(5,35).plot.hist(bins=50,ax=axes[0,0],color=ORANGE,edgecolor='none',alpha=0.85)
axes[0,0].axvline(players['player_efficiency_rating'].mean(),color=RED,linewidth=2,linestyle='--',
    label=f"Mean: {players['player_efficiency_rating'].mean():.1f}")
axes[0,0].axvline(15,color=GOLD,linewidth=1.5,linestyle=':',label='League avg (15)')
axes[0,0].set_title('PER Distribution',fontweight='bold',color='white'); axes[0,0].legend(fontsize=9)

sample=players[players['games_played']>=40].sample(min(4000,len(players)))
sc=axes[0,1].scatter(sample['points_per_game'],sample['win_shares'],
    c=sample['season'],cmap='plasma',alpha=0.25,s=10,edgecolors='none')
plt.colorbar(sc,ax=axes[0,1],label='Season')
axes[0,1].set_title('PPG vs Win Shares',fontweight='bold',color='white')

sns.boxplot(data=players[players['games_played']>=30],x='position',y='box_plus_minus',
    order=['PG','SG','SF','PF','C'],palette=POS_COLORS,ax=axes[1,0],linewidth=1.0)
axes[1,0].axhline(0,color='white',linestyle='--',alpha=0.4)
axes[1,0].set_title('Box Plus/Minus by Position',fontweight='bold',color='white')

top_vorp=players.groupby('player_name')['vorp'].sum().nlargest(15).sort_values()
top_vorp.plot.barh(ax=axes[1,1],color=GREEN,edgecolor='none',alpha=0.85)
axes[1,1].set_title('Career VORP Leaders',fontweight='bold',color='white')

plt.tight_layout(); plt.show()

## 7. Salary Inflation

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,6))

sal_yr=players[players['games_played']>=20].groupby('season')['salary_million_usd'].agg(['mean','median']).reset_index()
axes[0].plot(sal_yr['season'],sal_yr['mean'],color=GOLD,linewidth=2.2,label='Mean')
axes[0].plot(sal_yr['season'],sal_yr['median'],color=GREEN,linewidth=2,linestyle='--',label='Median')
axes[0].fill_between(sal_yr['season'],sal_yr['median'],sal_yr['mean'],alpha=0.15,color=GOLD)
axes[0].set_title('NBA Salary Inflation (1980–2026)',fontweight='bold',color='white')
axes[0].set_ylabel('Salary ($M)'); axes[0].legend(fontsize=9)

sample2=players[players['games_played']>=40].sample(min(3000,len(players)))
axes[1].scatter(sample2['points_per_game'],sample2['salary_million_usd'],
    c=sample2['season'],cmap='viridis',alpha=0.2,s=12,edgecolors='none')
corr_sal=sample2['points_per_game'].corr(sample2['salary_million_usd'])
axes[1].set_title(f'PPG vs Salary (r={corr_sal:.3f})',fontweight='bold',color='white')
axes[1].set_xlabel('Points Per Game'); axes[1].set_ylabel('Salary ($M)')
plt.tight_layout(); plt.show()

## 8. The 3-Point Revolution

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,6))

three_yr=teams.groupby('season')['three_point_attempt_pct'].mean()
axes[0].plot(three_yr.index,three_yr.values*100,color=ORANGE,linewidth=2.5,marker='o',markersize=3)
axes[0].fill_between(three_yr.index,three_yr.values*100,alpha=0.15,color=ORANGE)
axes[0].set_title('3-Point Attempts as % of All FGA (1980–2026)',fontsize=13,fontweight='bold',color='white')
axes[0].set_ylabel('% of FGA')

axes[1].scatter(teams['three_point_attempt_pct']*100,teams['net_rating'],
    c=[list(ERA_COLORS.keys()).index(e) if e in ERA_COLORS else 0 for e in teams['era']],
    cmap='tab10',alpha=0.5,s=20,edgecolors='none')
corr_3=teams['three_point_attempt_pct'].corr(teams['net_rating'])
axes[1].set_title(f'3P Attempt Rate vs Net Rating (r={corr_3:.3f})',fontweight='bold',color='white')
axes[1].set_xlabel('3P Attempt % of FGA'); axes[1].set_ylabel('Net Rating')
axes[1].grid(True,alpha=0.15)
plt.tight_layout(); plt.show()

## 9. Position Analysis

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(16,10))
pos_order=['PG','SG','SF','PF','C']
p30=players[players['games_played']>=30]
pos_avg=p30.groupby('position').agg(pts=('points_per_game','mean'),ast=('assists_per_game','mean'),
    reb=('rebounds_per_game','mean'),stl=('steals_per_game','mean'),blk=('blocks_per_game','mean')).reindex(pos_order)

pos_avg[['pts','ast','reb']].plot.bar(ax=axes[0,0],color=[ORANGE,BLUE,GREEN],edgecolor='none',alpha=0.9)
axes[0,0].set_title('Avg Stats by Position',fontweight='bold',color='white')
axes[0,0].tick_params(axis='x',rotation=0); axes[0,0].legend(fontsize=9)

pos_avg[['stl','blk']].plot.bar(ax=axes[0,1],color=[TEAL,RED],edgecolor='none',alpha=0.9)
axes[0,1].set_title('Avg Steals & Blocks by Position',fontweight='bold',color='white')
axes[0,1].tick_params(axis='x',rotation=0); axes[0,1].legend(fontsize=9)

pos_yr=players.groupby(['season','position']).size().unstack(fill_value=0)
pos_yr_pct=pos_yr.div(pos_yr.sum(axis=1),axis=0)*100
pos_yr_pct[pos_order].plot.area(ax=axes[1,0],color=[BLUE,ORANGE,RED,GREEN,PURPLE],alpha=0.8,linewidth=0)
axes[1,0].set_title('Position Share Over Time (%)',fontweight='bold',color='white')
axes[1,0].legend(fontsize=8,ncol=5)

sal_pos=p30.groupby(['era','position'])['salary_million_usd'].mean().unstack()
sal_pos.plot.bar(ax=axes[1,1],color=[BLUE,ORANGE,RED,GREEN,PURPLE],edgecolor='none',alpha=0.9)
axes[1,1].set_title('Avg Salary by Position & Era ($M)',fontweight='bold',color='white')
axes[1,1].tick_params(axis='x',rotation=35); axes[1,1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## 10. All-Star Intelligence

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,7))

top_allstar=careers.nlargest(20,'all_star_selections').sort_values('all_star_selections')
axes[0].barh(top_allstar['player_name'],top_allstar['all_star_selections'],color=GOLD,edgecolor='none',alpha=0.9)
axes[0].set_title('All-Time All-Star Selection Leaders',fontsize=13,fontweight='bold',color='white')

allstar_comp=pd.DataFrame({
    'All-Stars': players[players['all_star']==1][['points_per_game','player_efficiency_rating','win_shares']].mean(),
    'Non All-Stars': players[players['all_star']==0][['points_per_game','player_efficiency_rating','win_shares']].mean()
})
allstar_comp.plot.bar(ax=axes[1],color=[GOLD,SMOKE],edgecolor='none',alpha=0.9,width=0.5)
axes[1].set_title('All-Star vs Non-All-Star Avg Stats',fontsize=13,fontweight='bold',color='white')
axes[1].tick_params(axis='x',rotation=0); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## 11. Conference & Championship Trends

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,6))

conf_yr=teams.groupby(['season','conference'])['wins'].mean().unstack()
conf_yr['East'].plot(ax=axes[0],color=BLUE,linewidth=2,label='East')
conf_yr['West'].plot(ax=axes[0],color=RED,linewidth=2,label='West')
axes[0].axhline(41,color='white',linestyle=':',alpha=0.4)
axes[0].set_title('Avg Wins: East vs West Over Time',fontweight='bold',color='white')
axes[0].legend(fontsize=9)

champs=teams.groupby('team_name')['won_championship'].sum().sort_values(ascending=True).tail(15)
colors_ch=[GOLD if v>=5 else ORANGE if v>=3 else GREEN for v in champs.values]
champs.plot.barh(ax=axes[1],color=colors_ch,edgecolor='none',alpha=0.9)
axes[1].set_title('Championship Wins by Franchise (1980–2026)',fontweight='bold',color='white')
plt.tight_layout(); plt.show()

## 12. 🤖 Win % Predictor

In [ ]:
m=teams.copy()
for col in ['team_id','conference','division','era']:
    m[col+'_enc']=LabelEncoder().fit_transform(m[col].astype(str))

feats=['season','conference_enc','division_enc','era_enc','offensive_rating','defensive_rating',
       'net_rating','pace','three_point_attempt_pct','true_shooting_pct','payroll_million_usd']
X=m[feats].values; y=m['win_pct'].values
kf=KFold(n_splits=5,shuffle=True,random_state=42)

for name,model in [
    ('Random Forest',     RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1)),
    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=200,max_depth=4,random_state=42))]:
    r2=cross_val_score(model,X,y,cv=kf,scoring='r2')
    mae=-cross_val_score(model,X,y,cv=kf,scoring='neg_mean_absolute_error')
    print(f"{name:25s}  R²={r2.mean():.4f}±{r2.std():.4f}  MAE={mae.mean():.4f}")

In [ ]:
gb=GradientBoostingRegressor(n_estimators=200,max_depth=4,random_state=42)
gb.fit(X,y)
fi=pd.Series(gb.feature_importances_,index=feats).sort_values()
fig,ax=plt.subplots(figsize=(10,6))
fi.plot.barh(color=[RED if v>0.1 else ORANGE if v>0.05 else BLUE for v in fi.values],edgecolor='none',ax=ax,alpha=0.9)
ax.set_title('Feature Importance — Win % Predictor',fontsize=13,fontweight='bold',color='white')
ax.set_xlabel('Relative Importance'); ax.grid(True,alpha=0.15)
plt.tight_layout(); plt.show()
print("\n🏀 Net Rating dominates — offense + defense in one number explains most of winning.")

## 📋 Key Findings

- **3-point attempts** grew from ~3% to 40%+ of FGA — the single biggest tactical shift
- **Pace** dropped in the Jordan era (defensive focus) and rebounded in the Curry era
- **LeBron** has the longest sustained prime, **Jokic** has the highest PER-per-minute of any modern center
- **Net Rating** explains ~85% of win % variance — it's the best single team metric
- **Salary** grew ~40x in real terms from 1980–2026; PPG correlates at r≈0.55
- **West** has been consistently stronger than **East** since 2000

---
*If this was useful, please upvote! 🙏*